# rl-neural-v1 -- GPU-trained neural Q ensemble on all training seeds

Runs `train_cluster_rl.py --learner neural` (repo root). It extracts `cluster-rl-training.zip` to
`~/neural-rl/cluster-rl-training`, overlays `neural-rl/` onto it and trains the **neural** variant:

- Games are unchanged: full normal games on the stock simulator via HTTP `/predict`, the hash-checked
  eat/rest baseline, the same protected modes, four search-pace options and rewards as the tabular learner.
- The 7-bit Q table is replaced by an ensemble of small MLPs over 18 continuous features
  (energy, age, speed, vision, biome, tree/agent distances and counts, fruitless time, scan cooldown,
  clock, population, meals). Inference is pure Python, so the exported agent needs no torch/numpy.
- After every round the **GPU** fits the ensemble (double-DQN targets) on the transitions of *all* rounds
  so far. Round 0 explores uniformly at random (untrained network); later rounds are epsilon-greedy.
- Frozen evaluation leaves the baseline only when the ensemble-mean advantage minus 0.5 x ensemble
  spread is at least the 0.1 margin. Each round's checkpoint is validated against the baseline.

The games are CPU-bound (`WORKERS` concurrent games); the GPU part takes seconds to minutes per round.
Rerunning the training cell resumes. Test seeds (3000:3032) are never touched here.

**Cluster setup:** same as `test-eat-rest-v1.ipynb` -- a git-ignored `.env` with
`GITHUB_TOKEN=<token>` in the kernel's starting directory.

In [ ]:
import os

CLONE_DIR = "/home/jovyan/Nordic-AI-cup-2026"
if os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    print(f"{CLONE_DIR} already cloned - skipping (use `git pull` there to update)")
else:
    # GitHub token is read from a git-ignored .env (GITHUB_TOKEN=...) in the kernel's cwd, or from the environment
    if os.path.isfile(".env"):
        for line in open(".env"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#"):
                os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
    TOKEN = os.environ.get("GITHUB_TOKEN")
    if not TOKEN:
        raise RuntimeError(f"GITHUB_TOKEN not set - create {os.path.abspath('.env')} containing GITHUB_TOKEN=<token>")
    !git clone https://{TOKEN}@github.com/sjoeen/Nordic-AI-cup-2026.git {CLONE_DIR}

In [ ]:
import os
import subprocess
import sys

os.chdir(CLONE_DIR)
!git fetch origin challenge-1V2
!git checkout challenge-1V2
!git pull origin challenge-1V2

print("cwd:", os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
import time

TRAIN_SEEDS = "1000:1256"        # start-inclusive, stop-exclusive: all 256 package-default training seeds
VALIDATION_SEEDS = "2000:2016"
TEST_SEEDS = "3000:3032"         # reserved, not run by this notebook
ROUNDS = 10                      # total rounds incl. completed ones (256 games each); raise and rerun to continue
WORKERS = 0                      # 0 = auto: min(container CPU limit, (free RAM - 2 GiB) / 2 GiB per game); games are CPU-bound, the GPU only trains the network
RUN = "runs/neural-v1"           # fixed once initialized; use a new name for different seeds/network/code
HIDDEN, MEMBERS = 32, 5          # MLP width (2 hidden layers) and ensemble size; pure-Python inference cost grows with both
STEPS, BATCH = 20000, 4096       # GPU gradient steps per round and minibatch per ensemble member

cmd = [sys.executable, "-u", "train_cluster_rl.py", "--learner", "neural", "--run", RUN, "--train-seeds", TRAIN_SEEDS,
       "--validation-seeds", VALIDATION_SEEDS, "--test-seeds", TEST_SEEDS,
       "--rounds", str(ROUNDS), "--workers", str(WORKERS),
       "--init-args", "--hidden", str(HIDDEN), "--members", str(MEMBERS), "--steps", str(STEPS), "--batch", str(BATCH)]
print("running:", " ".join(cmd))

t0 = time.perf_counter()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
elapsed = time.perf_counter() - t0

print(f"finished in {elapsed / 60:.1f} min, exit code {proc.returncode}")
if proc.returncode != 0:
    raise RuntimeError(f"train_cluster_rl.py failed with exit code {proc.returncode}")

In [ ]:
import json
import pandas as pd

RUN_DIR = os.path.join(os.path.expanduser("~"), "neural-rl", "cluster-rl-training", RUN)
summaries = json.load(open(os.path.join(RUN_DIR, "validation_summary.json")))

# One row per checkpoint: frozen validation score vs the baseline on the same seeds
per_round = pd.DataFrame([dict(checkpoint=name, mean=s["mean"], median=s["median"], minimum=s["minimum"],
                               full_horizon=s["full_horizon"], baseline_mean=s["paired"]["baseline_mean"],
                               mean_delta=s["paired"]["mean_delta"], seeds_better=s["paired"]["seeds_better"])
                          for name, s in sorted(summaries.items())])
per_round

In [ ]:
# Per-seed paired table for the checkpoint with the best validation mean
best = per_round.sort_values("mean", ascending=False).iloc[0]["checkpoint"]
s = summaries[best]
df = pd.DataFrame({"neural": s["seeds"], "delta_vs_baseline": s["paired"]["differences"]})
df.index.name = "seed"
df["baseline"] = df["neural"] - df["delta_vs_baseline"]
print("best on validation:", best)
df[["baseline", "neural", "delta_vs_baseline"]]

In [ ]:
# Per-round training survival and GPU trainer stats (training games explore, so survival here is NOT evidence of improvement)
import glob
rows = []
for path in sorted(glob.glob(os.path.join(RUN_DIR, "checkpoints", "round_*.json"))):
    ck = json.load(open(path))
    s = pd.Series([e["survival"] for e in ck["episodes"]])
    rows.append(dict(round=ck["round"], games=len(s), mean=s.mean(), median=s.median(), min=s.min(),
                     new_transitions=ck["added_transitions"], replayed=ck["replay"]["transitions"],
                     device=ck["replay"]["device"], train_seconds=ck["replay"]["seconds"], loss=ck["replay"]["final_loss"]))
pd.DataFrame(rows)